# Mask Filling – Wie vervollständigt ein KI-Sprachmodell Sätze?

## Was ist Mask Filling?

Große Sprachmodelle wie **XLM-RoBERTa** wurden trainiert, indem man ihnen Millionen von Texten gezeigt hat – mit einem Trick: Einzelne Wörter wurden dabei zufällig versteckt ("maskiert"), und das Modell musste lernen, die fehlenden Wörter vorherzusagen.

Genau dieses Prinzip kannst du hier selbst ausprobieren: Du gibst einen Satz ein, in dem ein Wort durch `<mask>` ersetzt ist – und das Modell schlägt vor, welche Wörter dort am wahrscheinlichsten passen.

**Verwendetes Modell:** [XLM-RoBERTa large](https://huggingface.co/FacebookAI/xlm-roberta-large) (entwickelt von Meta AI) – ein mehrsprachiges Modell, das über 100 Sprachen beherrscht, darunter Deutsch, Englisch, Französisch und viele mehr.

## 1. Installation der benötigten Bibliotheken

Die folgenden zwei Zellen müssen nur **einmalig** ausgeführt werden. Sie installieren:
- **PyTorch** – eine weit verbreitete Bibliothek für maschinelles Lernen
- **Transformers** – die Hugging Face-Bibliothek, die fertig trainierte KI-Modelle direkt zugänglich macht

In [ ]:
# Nur einmalig ausführen!
pip install torch --index-url https://download.pytorch.org/whl/cpu

In [ ]:
# Nur einmalig ausführen!
pip install transformers

## 2. Bibliothek importieren und Modell laden

In [ ]:
from transformers import pipeline  # Importiert die pipeline-Funktion aus der transformers-Bibliothek

In [ ]:
# Das Modell wird beim ersten Aufruf heruntergeladen (kann einige Minuten dauern)
unmasker = pipeline("fill-mask", model="FacebookAI/xlm-roberta-large")

## 3. Erstes Beispiel – Mask Filling auf Deutsch

Mit dem Parameter `top_k=10` legen wir fest, dass das Modell die **10 wahrscheinlichsten Wörter** für die Lücke ausgeben soll.

Die Ausgabe ist eine Liste von Vorschlägen. Jeder Eintrag enthält:
- `token_str` – das vorgeschlagene Wort
- `score` – die Wahrscheinlichkeit (zwischen 0 und 1), mit der das Modell dieses Wort für richtig hält
- `sequence` – den vollständigen Satz mit dem eingesetzten Wort

In [ ]:
unmasker("Die Bayerische Staatskanzlei ist eine <mask>", top_k=10)

---
## Aufgaben

### Aufgabe 1 – Eigene Sätze ausprobieren

Ersetze in der Zelle unten den Beispielsatz durch eigene Sätze. Setze `<mask>` an verschiedene Stellen im Satz (Anfang, Mitte, Ende).

- Was fällt dir bei den Vorschlägen auf?
- Sind die Vorschläge immer sinnvoll? Wann nicht?

In [ ]:
# Aufgabe 1: Trage hier deinen eigenen Satz ein
mein_satz = "München ist die <mask> Bayerns."

unmasker(mein_satz, top_k=10)

### Aufgabe 2 – Mehrsprachigkeit testen

XLM-RoBERTa ist ein **mehrsprachiges** Modell. Teste den gleichen Sachverhalt in verschiedenen Sprachen.

- Unterscheiden sich die Vorschläge je nach Sprache?
- Was könnte der Grund dafür sein?

In [ ]:
# Aufgabe 2: Derselbe Satz in verschiedenen Sprachen
print("Deutsch:")
print(unmasker("Berlin ist die <mask> Deutschlands.", top_k=5))

print("\nEnglisch:")
print(unmasker("Berlin is the <mask> of Germany.", top_k=5))

print("\nFranzösisch:")
print(unmasker("Berlin est la <mask> de l'Allemagne.", top_k=5))

### Aufgabe 3 – Wahrscheinlichkeiten verstehen

Der `score`-Wert gibt an, wie sicher das Modell bei einem Vorschlag ist.

- Vergleiche zwei Sätze: einen mit einer eindeutigen Lücke und einen mit einer offenen Lücke.
- Wann sind die Scores hoch (das Modell ist sich sicher), wann niedrig (das Modell ist unsicher)?
- Was sagt das über die Komplexität von Sprache aus?

In [ ]:
# Aufgabe 3: Sicherer vs. unsicherer Kontext
print("Eindeutiger Kontext (hohe Scores erwartet):")
for ergebnis in unmasker("Wasser besteht aus <mask> und Sauerstoff.", top_k=5):
    print(f"  {ergebnis['token_str']:15s}  Score: {ergebnis['score']:.4f}")

print("\nOffener Kontext (niedrige Scores erwartet):")
for ergebnis in unmasker("Das Wetter heute ist <mask>.", top_k=5):
    print(f"  {ergebnis['token_str']:15s}  Score: {ergebnis['score']:.4f}")

### Aufgabe 4 – Bias in Sprachmodellen erkennen

Sprachmodelle lernen aus echten Texten aus dem Internet – und diese Texte spiegeln gesellschaftliche Vorurteile wider.

- Vergleiche die Vorschläge für die beiden Sätze unten.
- Was fällt dir auf? Wie lässt sich das erklären?
- Welche Konsequenzen könnte das haben, wenn solche Modelle in der Praxis eingesetzt werden?

In [ ]:
# Aufgabe 4: Bias-Test
print("Satz A:")
for ergebnis in unmasker("Die Ärztin ist sehr <mask>.", top_k=5):
    print(f"  {ergebnis['token_str']:15s}  Score: {ergebnis['score']:.4f}")

print("\nSatz B:")
for ergebnis in unmasker("Der Arzt ist sehr <mask>.", top_k=5):
    print(f"  {ergebnis['token_str']:15s}  Score: {ergebnis['score']:.4f}")

print("\nSatz C:")
for ergebnis in unmasker("Die Krankenschwester ist sehr <mask>.", top_k=5):
    print(f"  {ergebnis['token_str']:15s}  Score: {ergebnis['score']:.4f}")